In [ ]:
# ==============================================================================
# FIGURE: Per-variable statistical fidelity (Credit dataset) - TVD for the
# discrete variables, KS for the continuous variables, CTGAN vs CTAB-GAN+.
#
# Regenerated from the saved CSVs only (no model training).
# Same logic as section 6.3-6.5 in credit_complete_pipeline.ipynb.
# No in-image title: the title belongs in the LaTeX \caption.
# ==============================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

DATA_DIR = Path("data")
FIG_DIR = Path("figures")
FIG_DIR.mkdir(exist_ok=True)

CATEGORICAL_COLS = ["SEX", "EDUCATION", "MARRIAGE"]
ORDINAL_COLS = ["PAY_1", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]
CONTINUOUS_COLS = [
    "LIMIT_BAL", "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
]
DISCRETE_COLS = CATEGORICAL_COLS + ORDINAL_COLS

real  = pd.read_csv(DATA_DIR / "real_minority_train.csv")
ctgan = pd.read_csv(DATA_DIR / "synthetic_ctgan_final.csv")
ctab  = pd.read_csv(DATA_DIR / "synthetic_ctab_final.csv")


def tvd(a, b):
    cats = sorted(set(a.dropna().unique()) | set(b.dropna().unique()))
    p = a.value_counts(normalize=True)
    q = b.value_counts(normalize=True)
    return 0.5 * sum(abs(p.get(c, 0.0) - q.get(c, 0.0)) for c in cats)


def ks(a, b):
    return stats.ks_2samp(a.dropna(), b.dropna()).statistic


rows = []
for col in DISCRETE_COLS:
    rows.append((col, "TVD", tvd(real[col].astype(int), ctgan[col].astype(int)),
                 tvd(real[col].astype(int), ctab[col].astype(int))))
for col in CONTINUOUS_COLS:
    rows.append((col, "KS", ks(real[col].astype(float), ctgan[col].astype(float)),
                 ks(real[col].astype(float), ctab[col].astype(float))))
fid = pd.DataFrame(rows, columns=["Variable", "Metric", "CTGAN", "CTAB-GAN+"]).set_index("Variable")

# --- Check: must reproduce the saved notebook output (run C) ---
expected = {"SEX": (0.1741, 0.0203), "MARRIAGE": (0.0677, 0.1364),
            "PAY_6": (0.1908, 0.1376), "LIMIT_BAL": (0.1569, 0.1628),
            "BILL_AMT4": (0.2506, 0.1937), "PAY_AMT6": (0.2201, 0.3155)}
ok = all(abs(fid.loc[v, "CTGAN"] - e[0]) < 5e-5 and abs(fid.loc[v, "CTAB-GAN+"] - e[1]) < 5e-5
         for v, e in expected.items())
print(fid.round(4).to_string())
print("\nOK: identical to the notebook (run C)." if ok
      else "\nWARNING: different from the notebook. Check the CSVs.")

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(18, 7),
                         gridspec_kw={"width_ratios": [len(DISCRETE_COLS), len(CONTINUOUS_COLS)]})
width = 0.35
panels = [(axes[0], fid[fid["Metric"] == "TVD"], "Discrete variables (TVD)"),
          (axes[1], fid[fid["Metric"] == "KS"],  "Continuous variables (KS statistic)")]
for ax, df, subtitle in panels:
    x = np.arange(len(df))
    ax.bar(x - width/2, df["CTGAN"],     width, label="CTGAN",     color="#FF4D4D", alpha=0.85)
    ax.bar(x + width/2, df["CTAB-GAN+"], width, label="CTAB-GAN+", color="#4D79FF", alpha=0.85)
    ax.axhline(0.05, color="orange", linestyle="--", linewidth=1, alpha=0.8, label="Good threshold (0.05)")
    ax.axhline(0.15, color="red",    linestyle="--", linewidth=1, alpha=0.6, label="Poor threshold (0.15)")
    ax.set_xticks(x)
    ax.set_xticklabels(df.index, rotation=45, ha="right", fontsize=10)
    ax.set_title(subtitle, fontsize=12)
ymax = 1.08 * fid[["CTGAN", "CTAB-GAN+"]].to_numpy().max()   # same scale on both panels, no clipping
for ax in axes:
    ax.set_ylim(0, ymax)
axes[0].set_ylabel("TVD / KS statistic (lower is better)")
axes[1].legend(fontsize=9, loc="upper left")

plt.tight_layout()
out = FIG_DIR / "fidelity_tvd_ks.png"
plt.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved: {out}")